[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/ab-test-practice/blob/main/assignments/week02_assignment_validity_threats.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 2 Assignment: Validity Threats in A/B Testing

**Context:** You are a Product Data Scientist at **FamilyNest** (Airbnb for families with young kids). Your PM, **PaM**, has asked you to analyze three experiments that ran over the past few weeks.

This week's focus: **validity threats** — assignment imbalance, incorrect targeting, and dimensional breakdowns.

You'll practice:
- Detecting assignment imbalance using chi-squared tests
- Identifying incorrect targeting / instrumentation issues
- Breaking down results by dimensions to uncover heterogeneous treatment effects
- Making recommendations even when experiments have problems

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency, chisquare
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def get_ci(series, confidence=0.95):
    """Calculate the confidence interval for a series."""
    n = len(series)
    mean = series.mean()
    se = series.std() / np.sqrt(n)
    z = stats.norm.ppf((1 + confidence) / 2)
    ci_lower = mean - z * se
    ci_upper = mean + z * se
    return mean, ci_lower, ci_upper

In [ ]:
def calculate_results(df, metric_col, group_col='group', control_label='control', treatment_label='treatment'):
    """Calculate A/B test results for a given metric."""
    control = df[df[group_col] == control_label][metric_col]
    treatment = df[df[group_col] == treatment_label][metric_col]
    
    control_mean, control_ci_lower, control_ci_upper = get_ci(control)
    treatment_mean, treatment_ci_lower, treatment_ci_upper = get_ci(treatment)
    
    # Difference
    diff = treatment_mean - control_mean
    se_diff = np.sqrt((control.std()**2 / len(control)) + (treatment.std()**2 / len(treatment)))
    z = stats.norm.ppf(0.975)
    ci_lower = diff - z * se_diff
    ci_upper = diff + z * se_diff
    
    # p-value (two-sided z-test for proportions)
    z_stat = diff / se_diff
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    # Relative lift
    relative_lift = diff / control_mean if control_mean != 0 else np.nan
    
    print(f"Control: {control_mean:.4f} [{control_ci_lower:.4f}, {control_ci_upper:.4f}] (n={len(control)})")
    print(f"Treatment: {treatment_mean:.4f} [{treatment_ci_lower:.4f}, {treatment_ci_upper:.4f}] (n={len(treatment)})")
    print(f"Difference: {diff:.4f} [{ci_lower:.4f}, {ci_upper:.4f}]")
    print(f"Relative Lift: {relative_lift:.2%}")
    print(f"P-value: {p_value:.4f}")
    print(f"Significant at alpha=0.05: {p_value < 0.05}")
    
    return {
        'control_mean': control_mean,
        'treatment_mean': treatment_mean,
        'diff': diff,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'p_value': p_value,
        'relative_lift': relative_lift
    }

In [ ]:
def custom_calculate_results(df, metric_col, group_col='group', 
                              control_label='control', treatment_label='treatment',
                              dimensions=None):
    """
    Calculate A/B test results with dimensional breakdowns.
    
    Returns a dict with:
      - 'overall': results for the full dataset
      - 'breakdown': dict of {dimension: {value: results}} for each dimension
    """
    results = {}
    
    # Overall results
    print("=" * 60)
    print("OVERALL RESULTS")
    print("=" * 60)
    results['overall'] = calculate_results(df, metric_col, group_col, control_label, treatment_label)
    
    # Dimensional breakdowns
    results['breakdown'] = {}
    if dimensions:
        for dim in dimensions:
            results['breakdown'][dim] = {}
            print(f"\n{'=' * 60}")
            print(f"BREAKDOWN BY: {dim.upper()}")
            print(f"{'=' * 60}")
            for value in sorted(df[dim].unique()):
                subset = df[df[dim] == value]
                print(f"\n--- {dim} = {value} (n={len(subset)}) ---")
                if len(subset[subset[group_col] == control_label]) > 0 and len(subset[subset[group_col] == treatment_label]) > 0:
                    results['breakdown'][dim][value] = calculate_results(
                        subset, metric_col, group_col, control_label, treatment_label
                    )
                else:
                    print("  Insufficient data for one or both groups.")
                    results['breakdown'][dim][value] = None
    
    return results

In [ ]:
def ab_test_chi2(data_column):
    """
    Perform a chi-squared test for assignment balance.
    
    Tests whether the observed group counts differ significantly 
    from what we'd expect under uniform random assignment.
    
    Parameters:
    -----------
    data_column : pd.Series
        The group assignment column (e.g., 'control' / 'treatment')
    
    Returns:
    --------
    p_value : float
    total_count : int
    observed_counts : pd.Series
    """
    if not isinstance(data_column, pd.Series):
        raise ValueError("The data_column must be a Pandas Series.")
    observed_counts = data_column.value_counts()
    total_count = observed_counts.sum()
    n_groups = len(observed_counts)
    expected_counts = [total_count / n_groups] * n_groups
    # Use scipy chi-squared goodness-of-fit test
    chi2_stat, p_value = chisquare(observed_counts.values, f_exp=expected_counts)
    return p_value, total_count, observed_counts

---
## Background

You've been at FamilyNest for a few months now. PaM has come to you with three experiments that need analysis. Each one has a twist that will test your ability to identify validity threats before drawing conclusions.

This week focuses on three key validity threats:
1. **Assignment Imbalance** — Are users split evenly between control and treatment?
2. **Incorrect Targeting** — Was the experiment applied to the right population?
3. **Dimensional Breakdowns** — Do results hold across subgroups, or is the effect driven by a specific segment?

### Metric Definitions
| Abbreviation | Metric | Description |
|---|---|---|
| NBL | New Bookings per Listing | Target metric — bookings received |
| NCL | New Cancellations per Listing | Guardrail metric — we don't want cancellations to increase |
| NAL | New Availability per Listing | Informative metric — calendar availability |

---
## Task 1: Default Open Calendar

### Context
Currently, when new hosts sign up on FamilyNest, their calendar starts **fully blocked**. They must manually open dates to receive bookings. PaM hypothesizes that this creates unnecessary friction — many hosts intend to be available but never get around to opening their calendar.

**Proposed Change:** Default new hosts to an **open calendar** (all dates available), and let them block dates they don't want.

### Experiment Details
- **Target Metric:** NBL (New Bookings per Listing)
- **Guardrail Metric:** NCL (New Cancellations per Listing) — if hosts have open calendars they didn't intend, cancellations might rise
- **Informative Metric:** NAL (New Availability per Listing)
- **Traffic:** ~200 new users/day
- **Desired MDE:** 10% relative lift on NBL
- **Baseline NBL rate:** 20%
- **Additional columns in dataset:** `continent`, `booked_previously`, `device`

### Task 1.a — Test Design

Calculate how long the test should run. Fill in the table below.

**Parameters:**
- Significance level (alpha): 0.05
- Power (1 - beta): 0.80
- Baseline rate: 20%
- MDE: 10% relative lift (i.e., detect a shift from 20% to 22%)
- Traffic: 200 new users/day (100 per variant in a 50/50 split)

| Metric | Baseline Rate | MDE (absolute) | Sample Size per Variant | Days Needed |
|--------|--------------|----------------|------------------------|-------------|
| NBL    | ?            | ?              | ?                      | ?           |

In [ ]:
# Task 1.a: Calculate required sample size and test duration
# Use statsmodels or a manual formula for sample size calculation
# 
# Hints:
# - For a proportion, the formula involves: z_alpha, z_beta, p, and delta
# - You can use: from statsmodels.stats.power import NormalIndPower or proportion_effectsize

# Your code here


<details><summary>💡 Hint: Sample Size Formula</summary>

For a two-proportion z-test:

```python
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

effect_size = proportion_effectsize(0.20, 0.22)  # baseline vs. baseline + MDE
power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.80, alternative='two-sided')
```

Then divide by users per variant per day to get days needed.
</details>

**Your Answer:**

| Metric | Baseline Rate | MDE (absolute) | Sample Size per Variant | Days Needed |
|--------|--------------|----------------|------------------------|-------------|
| NBL    | _fill in_    | _fill in_      | _fill in_              | _fill in_   |

### Task 1.b — Test Analysis

The experiment has run and the data is in. Let's analyze it step by step.

In [ ]:
# Load the data
df_calendar = pd.read_csv('../data/dataset_default_calendar.csv')
df_calendar.head()

In [ ]:
# Explore the dataset
print(f"Shape: {df_calendar.shape}")
print(f"\nColumns: {df_calendar.columns.tolist()}")
print(f"\nGroup distribution:")
print(df_calendar['group'].value_counts())
print(f"\nData types:")
print(df_calendar.dtypes)

#### Step 1: Check for Assignment Imbalance

Before analyzing results, we need to verify that randomization worked properly. Use the `ab_test_chi2()` function to check whether the control/treatment split is balanced.

In [ ]:
# Step 1: Check assignment imbalance
# Use ab_test_chi2() on the group column
# A p-value < 0.05 suggests significant imbalance

# Your code here


<details><summary>💡 Hint</summary>

```python
p_value, total, counts = ab_test_chi2(df_calendar['group'])
print(f"Chi-squared test p-value: {p_value:.4f}")
print(f"Total users: {total}")
print(f"Counts per group:\n{counts}")
```

If p > 0.05, assignment looks balanced and you can proceed with confidence.
</details>

**Your interpretation:** _Is the assignment balanced? Can we proceed?_

_Write your answer here_

#### Step 2: Analyze Target Metric (NBL)

In [ ]:
# Step 2: Analyze NBL (target metric)
# Use calculate_results() or custom_calculate_results()

# Your code here


**Your interpretation:** _Is the treatment effect on NBL statistically significant? What is the direction and magnitude?_

_Write your answer here_

#### Step 3: Check Guardrail Metric (NCL)

In [ ]:
# Step 3: Check guardrail metric (NCL)
# We DON'T want this to increase significantly

# Your code here


**Your interpretation:** _Is the guardrail metric safe? Are cancellations staying flat or decreasing?_

_Write your answer here_

#### Step 4: Check Informative Metric (NAL)

In [ ]:
# Step 4: Check informative metric (NAL)

# Your code here


**Your interpretation:** _What does the informative metric tell us about the mechanism of the treatment?_

_Write your answer here_

#### Step 5: Break Down by Dimensions

Now let's check if the treatment effect is consistent across different user segments. Use the `custom_calculate_results()` function to break down by `continent` and `booked_previously`.

In [ ]:
# Step 5: Dimensional breakdown
# Break down NBL results by continent and booked_previously

# Your code here


<details><summary>💡 Hint</summary>

```python
results = custom_calculate_results(
    df_calendar, 
    metric_col='NBL',  # or whatever the column is named
    dimensions=['continent', 'booked_previously']
)
```

Look at whether the lift is consistent across dimensions or driven by a specific segment.
</details>

**Your interpretation:** _Are the results consistent across dimensions? Is there any segment where the treatment doesn't work or works especially well?_

_Write your answer here_

#### Step 6: Summary and Recommendation

Write a summary for PaM. Include:
1. Was the experiment valid? (assignment balance)
2. Did the target metric improve?
3. Did the guardrail hold?
4. What did dimensional breakdowns reveal?
5. Your recommendation: Ship, Don't Ship, or Iterate?

**Your Summary and Recommendation:**

_Write your summary here_

- **Validity:** _..._
- **Target Metric (NBL):** _..._
- **Guardrail (NCL):** _..._
- **Dimensional Insights:** _..._
- **Recommendation:** _Ship / Don't Ship / Iterate_
- **Reasoning:** _..._

---
## Task 2: Email to Upload Photos

### Context
After onboarding, there's a significant drop-off at the "upload a photo" step of listing creation. PaM's team found that hosts who upload photos are 3x more likely to get their first booking. They want to send an **email reminder after 3 days** if no photo has been uploaded.

### Experiment Details
- **Target Metric:** NBL (New Bookings per Listing)
- **Traffic:** ~5,000 users/day
- **Desired MDE:** 5% relative lift on NBL
- **Baseline NBL rate:** 20%

### Task 2.a — Test Design

Calculate how long the test should run.

| Metric | Baseline Rate | MDE (absolute) | Sample Size per Variant | Days Needed |
|--------|--------------|----------------|------------------------|-------------|
| NBL    | ?            | ?              | ?                      | ?           |

In [ ]:
# Task 2.a: Calculate required sample size and test duration
# Traffic: 5,000 users/day (2,500 per variant)
# Baseline: 20%, MDE: 5% relative lift (20% -> 21%)

# Your code here


**Your Answer:**

| Metric | Baseline Rate | MDE (absolute) | Sample Size per Variant | Days Needed |
|--------|--------------|----------------|------------------------|-------------|
| NBL    | _fill in_    | _fill in_      | _fill in_              | _fill in_   |

### Task 2.b — Test Analysis

**IMPORTANT:** Before analyzing results, always check for assignment imbalance first! The chi-squared test may reveal something interesting here.

In [ ]:
# Load the data
df_upload_photos = pd.read_csv('../data/dataset_upload_photos.csv')
df_upload_photos.head()

In [ ]:
# Explore the dataset
print(f"Shape: {df_upload_photos.shape}")
print(f"\nColumns: {df_upload_photos.columns.tolist()}")
print(f"\nGroup distribution:")
print(df_upload_photos['group'].value_counts())
print(f"\nData types:")
print(df_upload_photos.dtypes)

#### Step 1: Check for Assignment Imbalance

⚠️ **Pay close attention here.** Run the chi-squared test and interpret the results carefully.

In [ ]:
# Step 1: Check assignment imbalance
# Use ab_test_chi2() on the group column

# Your code here


**Your interpretation:** _What does the chi-squared test tell you? Is there a problem?_

_Write your answer here_

#### Step 2: Investigate the Imbalance

If the assignment is imbalanced, we need to investigate **why**. Check the group distribution broken down by available dimensions (e.g., `device`, `continent`).

Think about: Could a bug have caused certain users to not receive the treatment?

In [ ]:
# Step 2: Investigate the source of imbalance
# Break down assignment by dimensions to find the cause
# Check: is the imbalance consistent across all segments, or concentrated in one?

# Your code here


<details><summary>💡 Hint: Investigating Imbalance</summary>

Try cross-tabulating group assignment with each dimension:

```python
# Check assignment balance by device
print(pd.crosstab(df_upload_photos['device'], df_upload_photos['group']))
print()

# Check assignment balance by continent
print(pd.crosstab(df_upload_photos['continent'], df_upload_photos['group']))
```

If the imbalance is concentrated in one device type, it might indicate a bug where the email feature only worked on certain devices (e.g., the email trigger wasn't implemented on iOS).
</details>

**Your interpretation:** _What caused the imbalance? What does this tell you about the experiment's validity?_

_Write your answer here_

#### Step 3: Decide Whether to Proceed

Given what you found, can you still trust the results? Consider:
- If the imbalance is due to a specific device not receiving treatment, can you subset to only the devices that were properly treated?
- If you do subset, check assignment balance again on the subset.

In [ ]:
# Step 3: If appropriate, subset the data and re-check balance
# Then proceed with analysis on the valid subset

# Your code here


#### Step 4: Analyze Results (on valid data)

Analyze the target metric on whichever subset you determined is valid.

In [ ]:
# Step 4: Analyze NBL on the valid subset

# Your code here


#### Step 5: Summary and Recommendation

**Your Summary and Recommendation:**

_Write your summary here_

- **Validity Issue Found:** _..._
- **Root Cause:** _..._
- **Salvageable?** _Yes/No — explain why_
- **Results (if analyzable):** _..._
- **Recommendation:** _Ship / Don't Ship / Iterate / Re-run_
- **Reasoning:** _..._

---
## Task 3: New Map Service for Africa

### Context
FamilyNest uses a third-party map API to help hosts locate their property address. The engineering team found a new map provider that has **much better coverage of African addresses**. They want to test switching to this new provider for users in Africa.

### Experiment Details
- **Target Metric:** NBL (New Bookings per Listing)
- **Traffic:** ~4,000 users/day
- **Desired MDE:** 5% relative lift on NBL
- **Baseline NBL rate:** 20%
- **Intended targeting:** Only users in Africa should be in this experiment

### Task 3.a — Test Design

Calculate how long the test should run.

| Metric | Baseline Rate | MDE (absolute) | Sample Size per Variant | Days Needed |
|--------|--------------|----------------|------------------------|-------------|
| NBL    | ?            | ?              | ?                      | ?           |

In [ ]:
# Task 3.a: Calculate required sample size and test duration
# Traffic: 4,000 users/day (2,000 per variant)
# Baseline: 20%, MDE: 5% relative lift (20% -> 21%)

# Your code here


**Your Answer:**

| Metric | Baseline Rate | MDE (absolute) | Sample Size per Variant | Days Needed |
|--------|--------------|----------------|------------------------|-------------|
| NBL    | _fill in_    | _fill in_      | _fill in_              | _fill in_   |

### Task 3.b — Test Analysis

**IMPORTANT:** This experiment was supposed to only target users in Africa. Check the data carefully to see if that's actually what happened.

In [ ]:
# Load the data
df_africa_map = pd.read_csv('../data/dataset_africa_map.csv')
df_africa_map.head()

In [ ]:
# Explore the dataset
print(f"Shape: {df_africa_map.shape}")
print(f"\nColumns: {df_africa_map.columns.tolist()}")
print(f"\nGroup distribution:")
print(df_africa_map['group'].value_counts())
print(f"\nContinent distribution:")
print(df_africa_map['continent'].value_counts())

#### Step 1: Check Assignment Imbalance

In [ ]:
# Step 1: Check assignment imbalance on the full dataset

# Your code here


#### Step 2: Check Targeting / Instrumentation

The experiment was supposed to only include African users. Let's verify this.

In [ ]:
# Step 2: Check if the experiment was correctly targeted
# Are there users from continents OTHER than Africa in the data?

# Your code here


<details><summary>💡 Hint: Checking Targeting</summary>

```python
# Check continent distribution in the experiment
print("Continent distribution:")
print(df_africa_map['continent'].value_counts())
print(f"\nPercentage from Africa: {(df_africa_map['continent'] == 'Africa').mean():.1%}")

# Cross-tab continent with group
print("\nContinent x Group:")
print(pd.crosstab(df_africa_map['continent'], df_africa_map['group']))
```

If you find users from other continents, the experiment was NOT correctly targeted. This is an instrumentation/targeting bug!
</details>

**Your interpretation:** _Was the experiment correctly targeted? What went wrong?_

_Write your answer here_

#### Step 3: Analyze the Full (Incorrectly Targeted) Data

First, let's see what the results look like on ALL users (to understand what we'd incorrectly conclude).

In [ ]:
# Step 3: Analyze NBL on the FULL dataset (all continents)
# This shows what we'd conclude if we didn't check targeting

# Your code here


#### Step 4: Subset to Correct Population and Re-analyze

Now subset to only African users (the intended target) and re-run the analysis.

In [ ]:
# Step 4: Subset to Africa only and re-analyze

# Your code here


<details><summary>💡 Hint: Subsetting</summary>

```python
# Subset to Africa
df_africa_only = df_africa_map[df_africa_map['continent'] == 'Africa'].copy()
print(f"Africa-only sample size: {len(df_africa_only)}")

# Re-check balance on the subset
p_value, total, counts = ab_test_chi2(df_africa_only['group'])
print(f"Chi-squared p-value (Africa only): {p_value:.4f}")

# Analyze NBL
calculate_results(df_africa_only, 'NBL')
```
</details>

#### Step 5: Compare Results

Compare the results from the full dataset vs. the correctly targeted subset. What's different?

In [ ]:
# Step 5: Compare full dataset results vs. Africa-only results
# What changes? Why does it matter?

# Your code here


#### Step 6: Summary and Recommendation

**Your Summary and Recommendation:**

_Write your summary here_

- **Targeting Issue:** _..._
- **Impact on Results:** _How did incorrect targeting affect the results?_
- **Correct Results (Africa only):** _..._
- **Recommendation:** _Ship / Don't Ship / Iterate / Re-run_
- **Reasoning:** _..._
- **Engineering Action Item:** _What should be fixed before re-running?_

---
## Bonus: Break into Two Pages

### Context
The onboarding form for new hosts is a single long page with many fields. PaM wants to test splitting it into **two shorter pages** to see if completion rates improve and ultimately lead to more bookings.

### Experiment Details
- **Target Metric:** NBL (New Bookings per Listing)
- **Traffic:** ~7,000 users/day
- **Desired MDE:** 5% relative lift on NBL
- **Baseline NBL rate:** 20%

### Bonus.a — Test Design

Calculate how long the test should run.

| Metric | Baseline Rate | MDE (absolute) | Sample Size per Variant | Days Needed |
|--------|--------------|----------------|------------------------|-------------|
| NBL    | ?            | ?              | ?                      | ?           |

In [ ]:
# Bonus.a: Calculate required sample size and test duration
# Traffic: 7,000 users/day (3,500 per variant)
# Baseline: 20%, MDE: 5% relative lift (20% -> 21%)

# Your code here


**Your Answer:**

| Metric | Baseline Rate | MDE (absolute) | Sample Size per Variant | Days Needed |
|--------|--------------|----------------|------------------------|-------------|
| NBL    | _fill in_    | _fill in_      | _fill in_              | _fill in_   |

### Bonus.b — Full Analysis

Perform the complete analysis workflow, including all validity checks you've learned:
1. Load and explore the data
2. Check assignment imbalance
3. Check targeting (are the right users in the experiment?)
4. Analyze target metric
5. Check guardrail metrics (if available)
6. Dimensional breakdowns
7. Summary and recommendation

In [ ]:
# Load the data
df_split_page = pd.read_csv('../data/dataset_split_page.csv')
df_split_page.head()

In [ ]:
# Explore the dataset
print(f"Shape: {df_split_page.shape}")
print(f"\nColumns: {df_split_page.columns.tolist()}")
print(f"\nGroup distribution:")
print(df_split_page['group'].value_counts())
print(f"\nData types:")
print(df_split_page.dtypes)

In [ ]:
# Step 1: Check assignment imbalance

# Your code here


In [ ]:
# Step 2: Check targeting — are the right users included?

# Your code here


In [ ]:
# Step 3: Analyze target metric (NBL)

# Your code here


In [ ]:
# Step 4: Check guardrail metrics (if available in the dataset)

# Your code here


In [ ]:
# Step 5: Dimensional breakdowns

# Your code here


**Your Summary and Recommendation:**

_Write your complete summary here, covering:_

1. **Validity Checks:**
   - Assignment balance: _..._
   - Targeting: _..._

2. **Results:**
   - Target metric (NBL): _..._
   - Guardrail metrics: _..._
   - Dimensional breakdowns: _..._

3. **Recommendation:** _Ship / Don't Ship / Iterate / Re-run_

4. **Reasoning:** _..._

---
## Reflection Questions

After completing all tasks, answer these reflection questions:

1. **Why is it important to check assignment balance before analyzing results?** What could go wrong if you skip this step?

2. **When you find an assignment imbalance, what are your options?** When can you still trust the results, and when should you discard them?

3. **How did incorrect targeting in Task 3 affect the results?** Would you have drawn the wrong conclusion without checking?

4. **What's the value of dimensional breakdowns?** Give an example from this assignment where they provided useful insight.

**Your Reflections:**

1. _..._

2. _..._

3. _..._

4. _..._

---
## Checklist

Before submitting, make sure you have:

- [ ] Completed Task 1 (Default Open Calendar) — design + full analysis
- [ ] Completed Task 2 (Email to Upload Photos) — identified and investigated imbalance
- [ ] Completed Task 3 (Africa Map) — identified targeting issue, analyzed correct subset
- [ ] Attempted the Bonus (Split Page) — full validity-check workflow
- [ ] Answered all interpretation questions in markdown cells
- [ ] Written clear recommendations for PaM for each experiment
- [ ] Completed the reflection questions